# BERTopic Modeling
- 01_preprocessing.ipynb 에서 작업한 cleaned_finance_ai_abstracts.csv 로 topic modeling 수행
- 실행 환경: Colab T4(Python 3)

# [0] Colab 환경 설정

## 0-1. 구글드라이브 마운트

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0-2. 패키지 설치
- 설치 후 런타임 재시작 및 모두 실행

In [2]:
!pip install bertopic
!pip install sentence-transformers
!pip install umap-learn
!pip install hdbscan


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.7 MB/s eta 0:00:00


## 0-3. 라이브러리 설정
약 1분 소요

In [3]:
import os
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from umap import UMAP
from hdbscan import HDBSCAN

## 0-4. 파일 경로 설정 및 데이터 로드

In [4]:
# 기본 경로
base_path = '/content/drive/MyDrive/Colab Notebooks/GenAI-Finance-TopicModeling/'

# 파일 경로
file_path = os.path.join(
    base_path,
    'output_01_cleaned_abstracts.csv'
)

# CSV 불러오기
article_df = pd.read_csv(file_path)

# 데이터 확인
print(article_df.shape)

article_df.head()

(602, 2)


,Title,Cleaned_Abstract
0,The Odyssey of robots.txt Governance: Measurin...,web content essential element model service su...
1,Evaluation of a Large Language Model on the Am...,backgroundlarge model llm including chatgpt ch...
2,HDLGen-ChatGPT Case Study: RISC-V Processor VH...,present open source hdlgen chatgpt application...
3,Experience with Large Language Model Applicati...,model llm offer promising capability informati...
4,Large Language Model Agents for Investment Man...,recent advance model llm triggered new wave in...


# [1] BERTopic Modeling

## 1-1. 문서 리스트 생성

In [5]:
# BERTopic 입력 문서
docs = article_df['Cleaned_Abstract'].tolist()

print(f"문서 개수: {len(docs)}")

문서 개수: 602


## 1-2. 모델 설계

In [6]:
# SentenceTransformer 모델
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# UMAP 모델: 차원 축소
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',     # 문장 임베딩에 적합
    random_state=42
)

# HDBSCAN 모델: 클러스터링
hdbscan_model = HDBSCAN(
    min_cluster_size=15,      # 602개 문헌: 5 → 너무 잘게 쪼개지고, 30 → 너무 큰 토픽만 남아서 최소 클러스터를 15로 설정
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# CountVectorizer 모델: BoW 벡터화
vectorizer_model = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 2),      # unigram과 bigram 모두 고려 -> "risk management"과 같은 단어를 topic keyword 포착 가능
    min_df=5     # 최소 5개 문헌에 등장하는 단어만 고려 -> 너무 희귀한 단어는 노이즈로 작용할 수 있기 때문에 제거
)

# c-tf-idf 모델: 토픽별 중요 단어 계산
ctfidf_model = ClassTfidfTransformer()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 1-3. BERTopic 모델 생성

In [7]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics=8,
    calculate_probabilities=True,
    verbose=True
)

## 1-4. 모델 학습


In [8]:
topics, probs = topic_model.fit_transform(docs)

2026-05-15 08:14:50,111 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

2026-05-15 08:14:52,605 - BERTopic - Embedding - Completed ✓
2026-05-15 08:14:52,607 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-15 08:15:02,192 - BERTopic - Dimensionality - Completed ✓
2026-05-15 08:15:02,193 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-15 08:15:02,239 - BERTopic - Cluster - Completed ✓
2026-05-15 08:15:02,240 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-05-15 08:15:02,388 - BERTopic - Representation - Completed ✓
2026-05-15 08:15:02,389 - BERTopic - Topic reduction - Reducing number of topics
2026-05-15 08:15:02,391 - BERTopic - Topic reduction - Number of topics (8) is equal or higher than the clustered topics(5).
2026-05-15 08:15:02,392 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-15 08:15:02,619 - BERTopic - Representation - Completed ✓


## 1-5. 결과 확인

In [9]:
# 토픽 확인
print(f"[토픽 정보]\n{topic_model.get_topic_info()}")

# 토픽 키워드 확인
print(f"[토픽 0 키워드]\n{topic_model.get_topic(0)}")

[토픽 정보]
   Topic  Count                                     Name  \
0     -1      2            -1_inference_use_type_provide   
1      0     18           0_inference_high_time_solution   
2      1     97          1_gpt_multiple_knowledge_result   
3      2     22  2_application_result_promising_generate   
4      3    463         3_task_application_learning_time   

                                      Representation  \
0  [inference, use, type, provide, high, learning...   
1  [inference, high, time, solution, demand, vari...   
2  [gpt, multiple, knowledge, result, overall, hi...   
3  [application, result, promising, generate, lea...   
4  [task, application, learning, time, knowledge,...   

                                 Representative_Docs  
0  [digital automated assessment valuable time ef...  
1  [model llm exploded popularity due new generat...  
2  [background artificial intelligence become tra...  
3  [blockchain technology exploded popularity pro...  
4  [featured applic

# [2] 시각화

## 2-1. Intertopic Distance Map

In [10]:
topic_model.visualize_topics()

## 2-2. Topic Word Scores

In [11]:
topic_model.visualize_barchart()

## 2-3. Topic Similarity Heatmap

In [12]:
topic_model.visualize_heatmap()